# Planet at the evaluation sites: SuperDove and Tanager

Searches Planet's catalog (STAC API, `planet` in `lib/stac_providers.yaml`) at every optical evaluation site, year by year, and
publishes the **Planet** tabs of the Vendor archive first look page.

* **Login:** the `[Planet_STAC_DATA_API]` section of `~/code/credentials.ini`. Planet takes the API key as the user name of a Basic login.
* **One acquisition** is one strip (`pl:strip_id`): a pass is delivered as short scenes, and the scenes of one strip at a site count once.
* **SuperDove** is `PSScene` from the SuperDove instrument (`PSB.SD`); older Doves are left out. All ~300 satellites show as one sensor.
* **Tanager** is `TanagerScene` (Tanager-1, hyperspectral).
* **Size:** SuperDove images nearly all land daily, about 30–45k scenes a year at these sites. A year takes ~15 minutes to search.
* CSDA's own PlanetScope holdings are not checked here.

In [ ]:
import sys, os, glob, warnings
os.environ['USE_PYGEOS'] = '0'
warnings.filterwarnings('ignore')
sys.path.insert(0, '../lib')
import pandas as pd, geopandas as gpd
import multistac, archive_dashboard

SITES = archive_dashboard.SITES   # csda_sites_aoi.geojson
WORK = '/explore/nobackup/people/pmontesa/multistac/site_analysis/planet_sites/'   # one file per year
COLLECTIONS = {'planet': ['PSScene', 'TanagerScene']}
YEARS = range(2020, pd.Timestamp.today().year + 1)
REFRESH = [pd.Timestamp.today().year]    # years searched again; earlier years are read from WORK
sites = archive_dashboard.load_sites()   # CSDA optical and thermal validation sites, as in every first-look tab

## 1. Search each year
Month by month. A busy month that fails is split in half and tried again.

In [ ]:
def search_year(y):
    end = min(pd.Timestamp(f'{y}-12-31'), pd.Timestamp.today()).strftime('%Y-%m-%d')
    r = multistac.search(sites, f'{y}-01-01', end, COLLECTIONS, split='month', progress=False)
    parts, still = [r.gdf], []
    def retry(site, a, b, depth):
        x = multistac.search(sites[sites.site_name.eq(site)], f'{a:%Y-%m-%d}', f'{b:%Y-%m-%d}', COLLECTIONS, progress=False)
        if not (x.log.status == 'failed').any():
            parts.append(x.gdf); return
        if depth >= 2 or a >= b:
            still.append(f'{site} {a:%Y-%m-%d}'); return
        m = a + (b - a) // 2
        retry(site, a, m, depth + 1); retry(site, m + pd.Timedelta(days=1), b, depth + 1)
    for _, row in r.log[r.log.status == 'failed'].iterrows():
        a, b = [pd.Timestamp(v[:10]) for v in row.interval.split('/')]
        retry(row.site_name, a, b, 0)
    g = pd.concat(parts, ignore_index=True).drop_duplicates(['site_name', 'id'])
    log = r.log.copy(); log.loc[log.status == 'failed', 'status'] = 'retried'
    gpd.GeoDataFrame(g, geometry='geometry', crs=r.gdf.crs).to_pickle(f'{WORK}{y}.pkl'); log.to_csv(f'{WORK}{y}_log.csv', index=False)
    print(y, len(g), 'scenes', g.groupby('collection').size().to_dict(), '| still failed:', still)

os.makedirs(WORK, exist_ok=True)
for y in YEARS:
    if y in REFRESH or not os.path.exists(f'{WORK}{y}.pkl'):
        search_year(y)

## 2. Publish the Planet tabs
Writes `archive_first_look/planet-*.json` and per-site footprints in the data repository (`../csda_summaries_data`), then rebuilds the page. Publish the data with `python lib/data_repo.py push`. SuperDove takes ~25 minutes (its footprints).

In [ ]:
years = sorted(glob.glob(WORK + '[0-9][0-9][0-9][0-9].pkl'))
g = pd.concat([pd.read_pickle(p) for p in years], ignore_index=True)
ps = g.collection.eq('PSScene')
g = g[~ps | g.instrument.astype(str).str.contains('PSB.SD', regex=False)].copy()     # SuperDove only
g.loc[g.collection.eq('PSScene'), ['constellation', 'sensor']] = ['SuperDove', 'SuperDove']
g.loc[g.collection.eq('TanagerScene'), ['constellation', 'sensor']] = ['Tanager', 'Tanager-1']
g = gpd.GeoDataFrame(g, geometry='geometry', crs='EPSG:4326')
log = pd.concat([pd.read_csv(p.replace('.pkl', '_log.csv')) for p in years], ignore_index=True)
print(len(g), g.constellation.value_counts().to_dict())
archive_dashboard.publish('planet', g, sites, SITES, 'Site Name', '2020-01-01', pd.Timestamp.today().strftime('%Y-%m-%d'), log)

## 3. Density maps
SuperDove density comes from Planet's statistics (counts per 0.5° land cell and month), Tanager's from its catalog:

```
python lib/archive_density.py fetch superdove
python lib/archive_density.py fetch tanager
python lib/archive_density.py build
```